# Development baseline benchmark

Run persistence and daily seasonal persistence through the shared Python runner. These are CPU comparisons without an optimizer. Only training and validation periods are scored; calibration and final test stay locked. The seasonal comparison uses explicitly matched cohorts.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import pandas as pd

here = Path.cwd().resolve()
repository = next(path for path in [here, *here.parents] if (path / 'docs/data/prepared-release.json').is_file())
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))
from ml.training.baselines import run_baselines

In [ ]:
run_id = datetime.now(timezone.utc).strftime('baselines-notebook-%Y%m%dT%H%M%S%fZ')
output = repository / 'ml/runs' / run_id
report = run_baselines(
    repository / 'ml/training/configs/do_pilot.json',
    output, repository=repository, matrix=True,
)

In [ ]:
metrics = pd.read_csv(output / 'metrics.csv')
metrics.loc[
    metrics['partition'].eq('validation')
    & metrics['subset'].eq('all_targets')
    & metrics['cohort'].isin(['all_horizon_specific', 'seasonal_matched']),
    ['configuration', 'horizon_hours', 'model', 'cohort', 'expected_windows',
     'valid_predictions', 'prediction_coverage', 'mae_mgL', 'rmse_mgL', 'mae_skill_vs_persistence'],
]